# Convert outputs to yearly zarr files

In [1]:
import os
import sys
import zarr
import yaml
from glob import glob
from datetime import datetime, timedelta

import numpy as np
import xarray as xr

In [2]:
config_name = os.path.realpath('verif_config.yml')

with open(config_name, 'r') as stream:
    conf = yaml.safe_load(stream)

In [3]:
model_name = 'swin-wrf'
source_dir = conf[model_name]['save_loc_gather']

In [4]:
year = 2020
filenames = sorted(glob(source_dir+f'{year}*.nc'))

In [5]:
fn = filenames[0]
ds = xr.open_dataset(fn)
ds = ds.drop_vars(['WRF_P', 'WRF_precip', 'WRF_PWAT', 'WRF_radar_composite', 'WRF_TCC', 'WRF_OLR'])

In [6]:
ds_std = xr.open_dataset('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/mean_std/C404_6h_std_1980_2019_16lev.nc')

In [7]:
fn_target = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/all_in_one/C404_GP_2020.zarr'
ds_target = xr.open_zarr(fn_target)[['WRF_precip', 'WRF_PWAT', 'WRF_radar_composite', 'WRF_TCC', 'WRF_OLR']]
ds_target_pick = ds_target.sel(time=ds['time'])

In [8]:
ds = ds.rename({'latitude': 'south_north', 'longitude': 'west_east', 'level': 'bottom_top'})
ds['bottom_top'] = ds_std['bottom_top']
ds['west_east'] = ds_target['west_east']
ds['south_north'] = ds_target['south_north']

In [9]:
ds_final = xr.merge([ds, ds_target_pick])

In [10]:
# zarr encodings
dict_encoding = {}
varnames = list(ds_final.keys())
varname_4D = ['WRF_U', 'WRF_V', 'WRF_T', 'WRF_Q']

chunk_size_3d = dict(chunks=(1, 336, 336))
chunk_size_4d = dict(chunks=(1, 16, 336, 336))
compress = zarr.Blosc(cname='zstd', clevel=1, shuffle=zarr.Blosc.SHUFFLE, blocksize=0)

for i_var, var in enumerate(varnames):
    if var in varname_4D:
        dict_encoding[var] = {'compressor': compress, **chunk_size_4d}
    else:
        dict_encoding[var] = {'compressor': compress, **chunk_size_3d}

In [1]:
# save_name = '/glade/campaign/ral/hap/ksha/DWC/GATHER/CONUS_GP_zarr/atmos_2020-01-01T00Z.zarr'
# ds_final.to_zarr(save_name, mode='w', consolidated=True, compute=True, encoding=dict_encoding)